<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [ ]:
# Corpus formado por pares: (clase del correo, texto del correo)
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

# Palabras que se usarán como variables del modelo
vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [ ]:
# Importar NumPy para trabajar con vectores y matrices
import numpy as np

# Ejemplo de un vector binario con una posición por palabra
x = np.array([1, 1, 1, 1, 1])

In [ ]:
# Convertir un texto en un vector binario según el vocabulario

def vectorizar(texto, vocabulario):
    # Inicialmente ninguna palabra del vocabulario está presente
    x = np.zeros(len(vocabulario))

    # Revisar cada palabra y marcar si aparece en el texto
    for i, palabra in enumerate(vocabulario):
        x[i] = int(palabra in texto)

    return x

In [ ]:
# Extraer únicamente los textos del corpus
textos = np.array([text[1] for text in corpus])

# Vectorizar cada correo usando las palabras del vocabulario
textos_vec = np.array([vectorizar(texto, vocabulario) for texto in textos])

# Mostrar la matriz: una fila por correo y una columna por palabra
textos_vec

array([[1., 1., 0., 0., 0.],
       [1., 1., 0., 0., 0.],
       [1., 1., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [1., 1., 0., 0., 1.],
       [0., 0., 1., 0., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.]])

**3. Validación manual**

Recordemos el vocabulario:

```[dinero, gratis, premio, proyecto, reunion]```

* _en la reunion habra dinero gratis_: esta es la entrada 4 del `corpus`, contiene las palabras "dinero", "gratis" y "reunion", no contiene las palabras "premio" ni "proyecto", por lo que so versión vectorizada sería `[1,1,0,0,1]`, veamos:

In [ ]:
# Mostrar un ejemplo del corpus y su representación vectorial
print(corpus[4])
print(textos_vec[4])

('spam', 'en la reunion habra dinero gratis')
[1. 1. 0. 0. 1.]


* _reunion mañana_: esta es la entrada 4 del `corpus`, contiene la palabra "reunion", no contiene ninguna de las demás palabras, por lo que so versión vectorizada sería `[0,0,0,0,1]`, veamos:

In [ ]:
# Mostrar un segundo ejemplo del corpus y su representación vectorial
print(corpus[8])
print(textos_vec[8])

('normal', 'reunion mañana')
[0. 0. 0. 0. 1.]


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [ ]:
# Identificar qué correos pertenecen a la clase spam
spam = np.array([element[0] == 'spam' for element in corpus])

# Contar el número total de correos spam
Ytot = np.sum(spam)

# Calcular el prior de spam y de la clase normal
Pspam = Ytot / len(corpus)
Pnormal = 1 - Pspam

print(f'La probabilidad de que un correo sea spam es: {Pspam * 100:.2f} %')
print(f'La probabilidad de que un correo NO sea spam es: {Pnormal * 100:.2f} %')

La probabilidad de que un correo sea spam es: 45.45 %
La probabilidad de que un correo NO sea spam es: 54.55 %


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [ ]:
# Mostrar la matriz de vectores binarios de todos los correos
textos_vec

array([[1., 1., 0., 0., 0.],
       [1., 1., 0., 0., 0.],
       [1., 1., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [1., 1., 0., 0., 1.],
       [0., 0., 1., 0., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.]])

In [ ]:
# Importar pandas para organizar las probabilidades en una tabla
import pandas as pd

# Crear una tabla: filas = palabras, columnas = clases
probs = pd.DataFrame(index=vocabulario)

# Separar los vectores según la clase del correo
textos_vec_spam = textos_vec[spam]
textos_vec_nospam = textos_vec[~spam]

# Calcular P(X_i=1 | clase) como la frecuencia de cada palabra
probs['Spam'] = np.array([
    np.sum(textos_vec_spam[:, i]) / len(textos_vec_spam)
    for i in range(len(vocabulario))
])
probs['NoSpam'] = np.array([
    np.sum(textos_vec_nospam[:, i]) / len(textos_vec_nospam)
    for i in range(len(vocabulario))
])

# Mostrar las probabilidades condicionales sin suavizado
probs

,Spam,NoSpam
dinero,0.8,0.000000
gratis,0.8,0.000000
premio,0.4,0.166667
proyecto,0.0,0.666667
reunion,0.2,0.500000


Si una palabra no aparece nunca en una clase, el modelo lo aprende y no podrá reconocer mensajes de spam que sí la contengan.

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [ ]:
# Crear una tabla para las probabilidades con suavizado de Laplace
probs_lap = pd.DataFrame(index=vocabulario)

# Aplicar Laplace: (número de apariciones + 1) / (número de correos + 2)
probs_lap['Spam'] = np.array([
    (np.sum(textos_vec_spam[:, i]) + 1) / (len(textos_vec_spam) + 2)
    for i in range(len(vocabulario))
])
probs_lap['NoSpam'] = np.array([
    (np.sum(textos_vec_nospam[:, i]) + 1) / (len(textos_vec_nospam) + 2)
    for i in range(len(vocabulario))
])

# Mostrar las probabilidades condicionales suavizadas
probs_lap

,Spam,NoSpam
dinero,0.714286,0.125
gratis,0.714286,0.125
premio,0.428571,0.250
proyecto,0.142857,0.625
reunion,0.285714,0.500


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [20]:
# Vector binario del correo nuevo: "dinero gratis"
x = np.array([1, 1, 0, 0, 0])

# Para cada palabra, elegir P(X_i=1 | clase) si aparece
# o P(X_i=0 | clase) = 1 - P(X_i=1 | clase) si está ausente
probabilidades_spam = np.where(
    x == 1,
    probs_lap['Spam'],
    1 - probs_lap['Spam']
)
probabilidades_nospam = np.where(
    x == 1,
    probs_lap['NoSpam'],
    1 - probs_lap['NoSpam']
)

# Multiplicar las probabilidades de las palabras
# usando la independencia condicional de Naive Bayes
likelihood_spam = np.prod(probabilidades_spam)
likelihood_nospam = np.prod(probabilidades_nospam)

# Calcular los priors de cada clase
prior_spam = Pspam
prior_nospam = 1 - Pspam

# Scores conjuntos: P(clase) * P(x | clase)
score_spam = prior_spam * likelihood_spam
score_nospam = prior_nospam * likelihood_nospam

# Normalizar los scores para obtener probabilidades posteriores
normalizador = score_spam + score_nospam
posterior_spam = score_spam / normalizador
posterior_nospam = score_nospam / normalizador

print(f'P(x | spam) = {likelihood_spam:.6f}')
print(f'P(x | normal) = {likelihood_nospam:.6f}')
print(f'S_spam = {score_spam:.6f}')
print(f'S_normal = {score_nospam:.6f}')
print(f'P(spam | x) = {posterior_spam:.6f}')
print(f'P(normal | x) = {posterior_nospam:.6f}')
print('Clase predicha:', 'spam' if posterior_spam > posterior_nospam else 'normal')

P(x | spam) = 0.178497
P(x | normal) = 0.002197
S_spam = 0.081135
S_normal = 0.001199
P(spam | x) = 0.985443
P(normal | x) = 0.014557
Clase predicha: spam


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

1. Al calcular la probabilidad conjunta bajo el modelo de Naive Bayes, estamos asumiendo que la probabilidad de que aparezca una palabra en el correo es independiente de que aparezca cualquier otra.
2. La independencia elimina el crecimiento exponencial de la cantidad de probabilidades, ya que solo se necesita la probabilidad de cada palabra en cada clase, esto es (5 palabras)$\times$(2 clases)=10 probabilidades.
3. Porque basándose en la forma en que calcula probabilidades, el modelo puede generar nuevos vectores que parezcan spam o no.